In [1]:
from __future__ import annotations

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from linearmodels.iv import IV2SLS

# DoubleML
try:
    import doubleml as dml
    from doubleml import DoubleMLPLIV
except ImportError:
    raise ImportError("Install DoubleML: pip install DoubleML")

# ML learners
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler

HMAX = 48

# Paths
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
FINAL = ROOT / 'data' / 'final'
RESULTS = ROOT / 'results' / 'dml'
RESULTS.mkdir(parents=True, exist_ok=True)

# Load clean matrix
df = pd.read_csv(FINAL / 'feature_matrix_INCREMENTAL_v1.csv', index_col=0, parse_dates=True)
df.index = df.index.to_period('M').to_timestamp('M')

print(f"Loaded: {df.shape}")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Columns: {len(df.columns)}")

Loaded: (385, 69)
Date range: 1990-02-28 to 2022-02-28
Columns: 69


In [2]:
# Saadaoui baseline controls (4)
BASE_CONTROLS = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']

# Our enriched macro controls (13 added, 2 dropped = 13 kept)
MACRO_CONTROLS = ['vix', 'gs10', 'tb3ms', 'tedrate', 'baa10y', 'us_spread',
                  'brent', 'gold', 'bdi', 'cny_usd', 'em_fx', 'reer', 'gscpi']

# NLP controls (4)
NLP_CONTROLS = ['gpr', 'ea_gpr', 'wui', 'gdelt_events']

# All controls
ALL_CONTROLS = BASE_CONTROLS + MACRO_CONTROLS + NLP_CONTROLS

print(f"Base controls: {len(BASE_CONTROLS)}")
print(f"Macro controls: {len(MACRO_CONTROLS)}")
print(f"NLP controls: {len(NLP_CONTROLS)}")
print(f"Total: {len(ALL_CONTROLS)}")

# Key variables
Y = 'lwti'      # outcome
D = 'lpri'      # treatment
Z = 'd2pri'     # instrument

Base controls: 4
Macro controls: 13
NLP controls: 4
Total: 21


In [3]:
def lp_iv(df, y, shock, controls, hmax=HMAX):
    """Linear LP-IV from Saadaoui replication."""
    results = {}
    for h in range(hmax + 1):
        df_h = df.copy()
        df_h['y_h'] = df_h[y].shift(-h)
        
        # Lagged controls
        lag_cols = []
        for l in range(1, 4):
            df_h[f'L{l}_{y}'] = df_h[y].shift(l)
            lag_cols.append(f'L{l}_{y}')
        for l in range(1, 3):
            df_h[f'L{l}_{shock}'] = df_h[shock].shift(l)
            lag_cols.append(f'L{l}_{shock}')
        
        exog = controls + lag_cols
        fdf = df_h[[shock, 'y_h', 'd2pri'] + exog].dropna()
        
        # IV2SLS
        mod = IV2SLS(fdf['y_h'], fdf[exog], fdf[shock], fdf['d2pri']).fit()
        results[h] = {
            'coef': mod.params[shock],
            'se': mod.std_errors[shock],
            'pval': mod.pvalues[shock]
        }
    return results

# Run baseline with different control sets
print("Running baseline LP-IV...")
baseline_base = lp_iv(df, Y, D, BASE_CONTROLS)
baseline_macro = lp_iv(df, Y, D, BASE_CONTROLS + MACRO_CONTROLS)
baseline_full = lp_iv(df, Y, D, ALL_CONTROLS)

print("Done.")

Running baseline LP-IV...


ValueError: regressors [exog endog] do not have full column rank